# PEFT full run

Operational runbook for the long thread that produces the thesis's **PEFT** val
numbers: the (r, lr, epoch) sweep, the confirmation pass, the freeze, and the
frozen `peft` condition scored on full val. Runs on an **A100**.

Like the AFSP runbook it is organised into **four phases**, because the COMET
stack and the generation stack cannot share one Python environment
(`requirements-comet.txt` pins `transformers==4.57.6` / `numpy==1.26.4`, the
generation stack pins `transformers==5.12.1` / `numpy==2.4.1`), and the pipeline's
own ordering forces the alternation:

| Phase | Runtime | What runs | Depends on |
|------|---------|-----------|------------|
| 1 | **generation** (`requirements.txt`) | `peft_sweep` — train the 3 remaining cells x 3 epochs, eval_loss pre-filter, generate + score val candidates | anchor cell (already trained) |
| 2 | **COMET** (`requirements-comet.txt`) | `peft_verify` (COMET + judge Φ), **freeze** the adapter | Phase 1 sweep result |
| 3 | **generation** (`requirements.txt`) | frozen `peft` condition on val + chrF/BLEU + stylometrics + judge Φ | frozen adapter |
| 4 | **COMET** (`requirements-comet.txt`) | `peft` COMET + paired bootstrap vs the ladder | Phase 3 outputs |


---
## Phase 1 — generation runtime · the sweep


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
if not os.path.isdir('Style-Aware-MT'):
    !git clone --branch feat/peft-implementation https://github.com/prnamhr/Style-Aware-MT.git
%cd Style-Aware-MT
!git rev-parse --short HEAD

In [ ]:
# Generation stack.
!pip install -r requirements.txt

!pip uninstall -y torchvision torchaudio

In [ ]:
import torch; print(torch.__version__, torch.cuda.is_available(), torch.version.cuda)

### Persist the expensive artifacts across sessions


In [ ]:
PERSIST = True
DRIVE_ROOT = '/content/drive/MyDrive/style-aware-mt/peft'

import os, pathlib, shutil
if PERSIST:
    from google.colab import drive
    drive.mount('/content/drive')
    for rel in ('models', 'outputs/peft_sweep', 'results_backup'):
        target = pathlib.Path(DRIVE_ROOT) / rel
        target.mkdir(parents=True, exist_ok=True)
        if rel == 'results_backup':
            continue                                  # backup dir only, not linked in
        link = pathlib.Path(rel)
        if link.is_symlink():
            print(f'{link} -> {link.resolve()} (already linked)')
            continue
        if link.exists():
            # outputs/peft_sweep is a real, committed dir. Move what is in it onto
            # Drive and replace it with the link, or nothing written there persists.
            moved = 0
            for item in link.iterdir():
                dest = target / item.name
                if not dest.exists():
                    shutil.move(str(item), str(dest))
                    moved += 1
            shutil.rmtree(link)
            print(f'{link}: moved {moved} existing file(s) onto Drive, replacing dir with link')
        link.parent.mkdir(parents=True, exist_ok=True)
        link.symlink_to(target, target_is_directory=True)
        print(f'{link} -> {target}')
    !ls -la models outputs/peft_sweep | head -20
else:
    print('PERSIST=False — artifacts live on the VM only; a session loss restarts Phase 1.')

### Preflight: clear the tiny-smoke artifacts out of the way

In [ ]:
import json, pathlib

VAL_N = sum(1 for line in open('data/splits/val.jsonl', encoding='utf-8') if line.strip())
print(f'full val = {VAL_N} segments\n')

for p in sorted(pathlib.Path('outputs/peft_sweep').glob('*_val.jsonl')):
    n = sum(1 for line in p.open(encoding='utf-8') if line.strip())
    if n < VAL_N:
        p.unlink()
        print(f'removed {p} ({n} rows — smoke/partial, would be reused as a candidate)')
    else:
        print(f'kept    {p} ({n} rows — full val)')

res = pathlib.Path('results/peft_sweep_val.json')
if res.exists():
    smoke = [c for c in json.loads(res.read_text(encoding='utf-8'))['cells'] if c['n'] < VAL_N]
    if smoke:
        res.unlink()
        print(f"removed {res} (smoke ranking over {[c['tag'] for c in smoke]}, n<{VAL_N})")
    else:
        print(f'kept    {res} (full-val ranking)')

### Restore the already-trained anchor cell

In [ ]:
import json, pathlib, shutil

# Point this at the restored copy (a Drive path, a tarball you unpack, ...) or set
# it to None if the anchor checkpoints are unavailable and you accept the retrain.
ANCHOR_SOURCE = f'{DRIVE_ROOT}/models/peft_lora_r16_lr2e-4' if PERSIST else None

# The recorded outcome of the multi-epoch run (peft_multiepoch_smoke_colab.ipynb).
ANCHOR_RECORDED = [(1, 679, 1.5333), (2, 1358, 1.5591), (3, 2037, 1.7799)]

anchor = pathlib.Path('models/peft_lora_r16_lr2e-4')
if ANCHOR_SOURCE and not (anchor / 'epoch_checkpoints.json').exists():
    src = pathlib.Path(ANCHOR_SOURCE)
    if src.exists() and src.resolve() != anchor.resolve():
        anchor.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(src, anchor, dirs_exist_ok=True)
        print(f'restored {src} -> {anchor}')
    elif not src.exists():
        print(f'ANCHOR_SOURCE {src} not found')

man_path = anchor / 'epoch_checkpoints.json'
if not man_path.exists():
    print('\nanchor NOT restored — the sweep will train r=16 lr=2e-4 from scratch (~3 h, 4 cells).')
else:
    man = json.loads(man_path.read_text(encoding='utf-8'))
    got = [(m['epoch'], m['step'], round(m['eval_loss'], 4)) for m in man]
    missing = [m['checkpoint'] for m in man if not m['checkpoint'] or not pathlib.Path(m['checkpoint']).exists()]
    print('restored anchor manifest:', got)
    assert got == ANCHOR_RECORDED, f'restored anchor does not match the recorded run {ANCHOR_RECORDED}'
    assert not missing, f'anchor checkpoint dir(s) missing on disk: {missing}'
    print('matches the recorded multi-epoch run; the sweep will skip this cell (3 left to train).')

### The grid plan


In [ ]:
!python manage.py peft_sweep --config configs/peft_sweep.yaml --dry-run

### Train, pre-filter, generate, score

In [ ]:
!python manage.py peft_sweep --config configs/peft_sweep.yaml --epochs-keep 2

### Manifest check on the newly trained cells


In [ ]:
import json
from pathlib import Path

ANCHOR_DIR = 'peft_lora_r16_lr2e-4'
manifests = sorted(Path('models').glob('peft_lora_*/epoch_checkpoints.json'))
print(f'{len(manifests)} cell manifest(s) found\n')
problems = []
for man_path in manifests:
    name = man_path.parent.name
    man = json.loads(man_path.read_text(encoding='utf-8'))
    epochs = [m['epoch'] for m in man]
    steps = [m['step'] for m in man]
    losses = [m['eval_loss'] for m in man]
    missing = [m['checkpoint'] for m in man if not m['checkpoint'] or not Path(m['checkpoint']).exists()]
    tag = '  (anchor, trained earlier)' if name == ANCHOR_DIR else ''
    print(f"{name:<26} epochs={epochs} steps={steps}{tag}")
    print('    eval_loss: ' + '  '.join(f'e{e}={l:.4f}' for e, l in zip(epochs, losses)))
    if name == ANCHOR_DIR:
        got = [(e, s, round(l, 4)) for e, s, l in zip(epochs, steps, losses)]
        if got != ANCHOR_RECORDED:
            problems.append(f'{name}: no longer matches the recorded run -> {got}')
        continue                      # property check already run in the multi-epoch notebook
    if epochs != [1, 2, 3]:
        problems.append(f'{name}: epochs not 1..3 -> {epochs}')
    if len(set(steps)) != len(steps):
        problems.append(f'{name}: duplicate checkpoint step -> {steps}')
    if len(set(round(l, 6) for l in losses)) <= 1:
        problems.append(f'{name}: eval_loss flat across epochs -> {losses}')
    if missing:
        problems.append(f'{name}: checkpoint dir(s) missing on disk -> {missing}')

assert len(manifests) == 4, f'expected 4 cells present, found {len(manifests)}'
assert not problems, 'manifest problems:\n  ' + '\n  '.join(problems)
print('\nMANIFESTS OK: 4 cells present, 3 newly trained ones checked, anchor unchanged.')

### The proxy pick


In [ ]:
import json

sweep = json.load(open('results/peft_sweep_val.json'))
print(f"epochs_keep={sweep['epochs_keep']}  adequacy_margin={sweep['adequacy_margin']}  "
      f"candidates={len(sweep['cells'])}\n")
hdr = f"{'tag':<24}{'r':>4}{'a':>5}{'lr':>9}{'ep':>4}{'n':>6}{'chrF':>8}{'reg_fit':>9}{'eval_loss':>11}"
print(hdr); print('-' * len(hdr))
for c in sorted(sweep['cells'], key=lambda c: c.get('register_fit', 1e9)):
    print(f"{c['tag']:<24}{c['r']:>4}{c['alpha']:>5}{c['lr']:>9g}{c['epoch']:>4}{c['n']:>6}"
          f"{c['chrF']:>8}{c.get('register_fit', float('nan')):>9}{c['eval_loss']:>11.4f}")

rec = sweep.get('recommended')
print('\nproxy recommended:', rec and {k: rec[k] for k in ('tag', 'r', 'lr', 'epoch', 'chrF', 'register_fit') if k in rec})

# Every candidate must have been generated on FULL val, not a leftover short file.
short = [c['tag'] for c in sweep['cells'] if c['n'] != VAL_N]
assert not short, f'candidates not scored on full val ({VAL_N}): {short}'

In [ ]:
# Back the phase-1 result up to Drive before switching runtimes.
if PERSIST:
    !cp -v results/peft_sweep_val.json {DRIVE_ROOT}/results_backup/

---
## Phase 2 — COMET runtime · verify + freeze


In [ ]:
!pip install -q -r requirements-comet.txt

In [ ]:
import os, getpass
if not os.environ.get('ANTHROPIC_API_KEY'):
    os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('ANTHROPIC_API_KEY: ')

In [ ]:
import logging
logging.getLogger("httpx").setLevel(logging.WARNING)

In [ ]:

!USE_TF=0 python -m src.peft.verify --config configs/peft_sweep.yaml \
    --judge-config configs/judge_eval.yaml --top 3

In [ ]:
# The freeze decision from the reported metrics.
import json

v = json.load(open('results/peft_verify_val.json'))
print('freeze tag :', v['freeze'],
      '(proxy pick held)' if v['proxy_pick_held'] else '(runner-up overtook the proxy pick)')
print('checkpoint :', v['freeze_checkpoint'], '\n')
for c in v['cells']:
    mark = '  <== freeze' if c['tag'] == v['freeze'] else ''
    phi = f"{c['judge_mean']:.3f}" if c['judge_mean'] is not None else 'n/a'
    print(f"  {c['tag']:<24} r={c['r']} lr={c['lr']:g} ep={c['epoch']}  "
          f"chrF {c['chrF']}  COMET {c['comet_system']:.4f}  Phi {phi}"
          f"  (judge cov {c['judge_coverage']}){mark}")

### Freeze the adapter into `configs/peft_qwen.yaml`

`generator.adapter_path` is what the `peft` inference condition loads. Point it
at the frozen checkpoint before generating the reported condition.

In [ ]:
import json, re, pathlib

v = json.load(open('results/peft_verify_val.json'))
ckpt = v['freeze_checkpoint']
assert ckpt and pathlib.Path(ckpt).exists(), f'frozen checkpoint missing on disk: {ckpt}'
frozen = next(c for c in v['cells'] if c['tag'] == v['freeze'])

p = pathlib.Path('configs/peft_qwen.yaml')
text = p.read_text(encoding='utf-8')
text, n = re.subn(r'(?m)^(\s*adapter_path:\s*)\S+', lambda m: f'{m.group(1)}{ckpt}', text, count=1)
assert n == 1, 'no generator.adapter_path line found in configs/peft_qwen.yaml'
p.write_text(text, encoding='utf-8')

print(f"froze generator.adapter_path = {ckpt}")
print(f"  (r={frozen['r']}, alpha={frozen['alpha']}, lr={frozen['lr']:g}, epoch={frozen['epoch']})")
!grep -n 'adapter_path:' configs/peft_qwen.yaml

In [ ]:
if PERSIST:
    !cp -v results/peft_verify_val.json {DRIVE_ROOT}/results_backup/

---
## Phase 3 — generation runtime · the frozen `peft` condition

In [ ]:
!pip install -q -r requirements.txt
# Same torchvision/torchaudio ABI mismatch as Phase 1 — drop them (text-only pipeline).
!pip uninstall -y torchvision torchaudio

In [ ]:
# Full val with the frozen adapter; resumable via outputs/peft_val.jsonl.
!python manage.py infer --condition peft --config configs/peft_qwen.yaml

In [ ]:
# Adequacy proxies (chrF/BLEU) and register stylometrics — free/local, no COMET.
!python manage.py eval         --conditions peft --split val
!python manage.py stylometrics --conditions peft --split val

In [ ]:
# Register fidelity (judge Phi) for the reported PEFT row.
import os, getpass
if not os.environ.get('ANTHROPIC_API_KEY'):
    os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('ANTHROPIC_API_KEY: ')
!python manage.py judge --conditions peft --split val --config configs/judge_eval.yaml

---
## Phase 4 — COMET runtime · PEFT COMET + paired bootstrap


In [ ]:
!pip install -q -r requirements-comet.txt

In [ ]:
!python manage.py comet --conditions peft --split val

In [ ]:
import pathlib

LADDER = ['zeroshot', 'random_fewshot', 'knn_fewshot', 'afsp_margin', 'afsp_full']
present = [c for c in LADDER if pathlib.Path(f'outputs/{c}_val.jsonl').exists()]
absent = [c for c in LADDER if c not in present]
print('ladder outputs present:', present)
if absent:
    print('MISSING (excluded from the bootstrap):', absent)

In [ ]:
# PEFT vs the ladder rungs that are present, baseline = knn_fewshot when available.
conds = ' '.join(['peft'] + present)
baseline = 'knn_fewshot' if 'knn_fewshot' in present else 'peft'
!python manage.py bootstrap --metric comet --conditions {conds} --split val --baseline {baseline}

In [ ]:
if PERSIST:
    !cp -v results/*val*.json {DRIVE_ROOT}/results_backup/ 2>/dev/null; ls {DRIVE_ROOT}/results_backup